## **1. Setup & Config**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

# Theme config
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

# Create figures dir
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Load config
CONFIG_PATH = Path("../configs/params.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

## **2. Load Data**
Merge train and store datasets. Filter open days with sales > 0.

In [ ]:
# Load raw
train_df = pd.read_csv(f"../{config['data']['raw_train']}", low_memory=False)
store_df = pd.read_csv(f"../{config['data']['raw_store']}")

# Merge
df_raw = pd.merge(train_df, store_df, on="Store", how="left")

# Base filter: Open and positive sales
df = df_raw[(df_raw["Open"] == 1) & (df_raw["Sales"] > 0)].copy()

# Fix types
df["Date"] = pd.to_datetime(df["Date"])
df["StateHoliday"] = df["StateHoliday"].astype(str)

print(f"Total entries (filtered): {len(df):,}")

## **3. Target Variable Analysis**
Analyze Sales distribution. Check log transform impact on skew.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw Sales
sns.histplot(df["Sales"], bins=50, kde=True, color="#4c72b0", ax=axes[0])
axes[0].set_title(f"Sales Distribution (Skew: {df['Sales'].skew():.2f})")

# Log Sales
log_sales = np.log1p(df["Sales"])
sns.histplot(log_sales, bins=50, kde=True, color="#55a868", ax=axes[1])
axes[1].set_title(f"Log(1+Sales) Distribution (Skew: {log_sales.skew():.2f})")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_target_distribution.png", dpi=300)
plt.show()

Raw sales heavily right-skewed. Log transform shape → near normal. Zero sales on open days negligible.

## **4. Temporal Trends**
Check Weekly patterns, monthly seasonality, DayOfWeek impact.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 12))

# Seasonal Trend
monthly_sales = (
    df.groupby(pd.Grouper(key="Date", freq="ME"))["Sales"].mean().reset_index()
)
sns.lineplot(
    data=monthly_sales, x="Date", y="Sales", marker="o", color="#c44e52", ax=axes[0]
)
axes[0].set_title("Average Sales Trend (Monthly)")
axes[0].set_ylabel("Avg Sales")

# DayOfWeek vs Promo
sns.barplot(
    data=df,
    x="DayOfWeek",
    y="Sales",
    hue="Promo",
    palette=["#4c72b0", "#dd8452"],
    ax=axes[1],
)
axes[1].set_title("Sales by Day of Week & Promo Status")

plt.tight_layout()
plt.savefig(FIG_DIR / "02_temporal_trends.png", dpi=300)
plt.show()

December spike clear. Promo drives massive volume jump across all days. DayOfWeek raw feature (1-7) distinct from dt.dayofweek (0-6). Mismatch = prediction noise.

## **5. Categorical Impact**
Evaluate StoreType, Assortment, Holidays.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# StoreType
sns.boxplot(
    data=df,
    x="StoreType",
    y="Sales",
    order=["a", "b", "c", "d"],
    ax=axes[0],
    palette="Blues",
)
axes[0].set_title("Sales by StoreType")

# Assortment
sns.boxplot(
    data=df,
    x="Assortment",
    y="Sales",
    order=["a", "b", "c"],
    ax=axes[1],
    palette="Greens",
)
axes[1].set_title("Sales by Assortment")

# StateHoliday
sns.boxplot(data=df, x="StateHoliday", y="Sales", ax=axes[2], palette="Reds")
axes[2].set_title("Sales by StateHoliday (a/b/c/0)")

plt.tight_layout()
plt.savefig(FIG_DIR / "03_categorical_impact.png", dpi=300)
plt.show()

StoreType 'b' massive outlier. Avg volume >10k. Assortment 'b' similarly high baseline. Need categorical mapping in engine.

## **6. Competition & Extended Promo Analysis**
Analyze distance, age of competition, Promo2 effect.

In [ ]:
# Compute Competition Age (Months)
comp_year = df["CompetitionOpenSinceYear"].fillna(1900).astype(int)
comp_month = df["CompetitionOpenSinceMonth"].fillna(1).astype(int)
df["CompAgeMonths"] = (df["Date"].dt.year - comp_year) * 12 + (
    df["Date"].dt.month - comp_month
)
df.loc[df["CompetitionOpenSinceYear"].isna(), "CompAgeMonths"] = 0
df["CompActive"] = (df["CompAgeMonths"] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Log(Distance)
temp_df = df.dropna(subset=["CompetitionDistance"]).sample(5000)
temp_df["LogDist"] = np.log1p(temp_df["CompetitionDistance"])
sns.scatterplot(data=temp_df, x="LogDist", y="Sales", alpha=0.2, ax=axes[0])
axes[0].set_title("Log(CompetitionDistance) vs Sales")

# Promo2 Impact
sns.barplot(data=df, x="Promo2", y="Sales", palette=["#4c72b0", "#dd8452"], ax=axes[1])
axes[1].set_title("General Sales: Promo2 vs No Promo2")

plt.tight_layout()
plt.savefig(FIG_DIR / "04_comp_promo2.png", dpi=300)
plt.show()

Log(Distance) fix long-tail skew. Active competition slightly suppress sales. Promo2 negatively correlate with volume. Defensive posture?

## **7. Correlation Matrix**
Quantify linear relations.

In [ ]:
# Extract basic temporal
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

cols = [
    "Sales",
    "Customers",
    "Promo",
    "DayOfWeek",
    "SchoolHoliday",
    "CompetitionDistance",
    "Year",
    "Month",
    "WeekOfYear",
]
corr = df[cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, cbar_kws={"shrink": 0.8}
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_correlation_matrix.png", dpi=300)
plt.show()

$\implies$ Promo dominate signal. Target log transform + categorical alignment + robust competition fill are main takeaways.